# 🛡️ WE3 — Adversarial Robustness & Certification
## Fool a network on purpose, then *prove* it can't be fooled

> **The scenario.** You ship the digit reader behind a bank's **automated cheque scanner**. On clean
> MNIST it is **99% accurate** — so good that nobody double-checks the amounts any more. Then a
> fraudster discovers something unsettling: by nudging a *handful of pixels* by an amount **no human
> eye can see**, they can make your network read a hand-written **7** as a **3** — or as *any digit
> they choose*. Same picture to you and me; a different answer from the model.
>
> This notebook lives on the two sides of that discovery.

| | **🗡️ Attack** | **🛡️ Certification** |
|---|---|---|
| **Goal** | *find one* input in the ε-ball that fools the model | *prove no* input in the ε-ball fools the model |
| **Finds** | a concrete counterexample | a mathematical guarantee |
| **Tells you** | an **upper bound** on robustness (it's *at least* this broken) | a **lower bound** on robustness (it's *at least* this safe) |
| **If it "fails"** | attack found nothing → says **nothing** about safety | box too wide → says **nothing** about danger |
| **Tools** | FGSM, targeted FGSM, **PGD** | interval bound propagation (**the box**) |

An attacker only needs to get **lucky once**; a defender has to be safe against **every** perturbation
at once. That asymmetry is why "we ran some attacks and found nothing" is *not* a safety proof — and
why certification exists.

**How this notebook works** — cells tagged **🔧 PROVIDED — run, don't edit** hand you the plumbing
(data, model, visuals). Cells tagged **🎯** build the real thing (attacks, defenses, the certifier).
Cells tagged **🧠** are quick gut-checks — answer before you reveal.

In [ ]:
#@title 🗺️ Roadmap — from one fooled pixel to a proof of safety { display-mode: "form" }
from IPython.display import HTML, display
_steps = [
    ("🗡️", "1 · FGSM",   "one gradient step",  "craft your first adversarial digit"),
    ("🎯", "2 · Targeted", "choose the lie",     "force any digit into any class you name"),
    ("🔁", "3 · PGD",     "many small steps",   "the strong attack: iterate inside the ε-ball"),
    ("🛡️", "4 · Defense",  "adversarial training","train a model that shrugs the attack off"),
    ("📐", "5 · Certify",  "push a whole box",   "*prove* no attack exists — interval bounds"),
]
_grad = ["#e0796d", "#db6fa9", "#9a63d4", "#667eea", "#39b36a"]
_b = ""
for (_ic, _t, _q, _d), _g in zip(_steps, _grad):
    _b += (f'<div class="rmr-step"><div class="rmr-ic" style="background:linear-gradient(135deg,{_g},{_g}cc)">{_ic}</div>'
           f'<div class="rmr-t">{_t}</div><div class="rmr-q">{_q}</div><div class="rmr-d">{_d}</div></div>'
           '<div class="rmr-ar">➜</div>')
_b = _b.rsplit('<div class="rmr-ar">➜</div>', 1)[0]
display(HTML(f'''
<style>
.rmr{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff;color:#2c2350}}
.rmr-h{{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}}
.rmr-s{{font-size:12px;color:#6b6685;margin:0 0 16px}}
.rmr-row{{display:flex;align-items:stretch;flex-wrap:wrap}}
.rmr-step{{flex:1 1 150px;min-width:140px;text-align:center;padding:0 6px}}
.rmr-ic{{width:52px;height:52px;border-radius:50%;margin:0 auto 8px;display:flex;align-items:center;justify-content:center;font-size:23px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}}
.rmr-t{{font-weight:800;font-size:13px;color:#2c2350}}
.rmr-q{{font-size:11px;color:#8b5cf6;margin-top:3px;font-weight:700}}
.rmr-d{{font-size:10.5px;color:#8b86a6;margin-top:5px;line-height:1.3}}
.rmr-ar{{display:flex;align-items:center;font-size:17px;color:#b9a9e6;flex:0 0 16px}}
</style>
<div class="rmr"><div class="rmr-h">🗺️ The plan</div>
<div class="rmr-s">Attacks get stronger left to right; then we <b>defend</b>, and finally <b>prove</b> what the attacks could only probe.</div>
<div class="rmr-row">{_b}</div></div>'''))

## 0. Setup

Four provided cells: install PyTorch, **fetch the pretrained models** from the course repo, define the
**visual helpers**, then load **MNIST** and the two networks you will attack and defend. Run them top
to bottom. The repo is **public**, so nothing needs a login or token.

**0.1 — 🔧 PROVIDED — fetch the repo** (it carries the two pretrained checkpoints in
`exercises/models/`). This cell is collapsed; just run it.

In [ ]:
#@title 🔧 PROVIDED — fetch the repo (run, don't edit) { display-mode: "form" }
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE3-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"   # public repo — no token needed
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding the `exercises/` helpers — so paths resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "exercises", "requirements_robustness.txt")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/requirements_robustness.txt) — re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these already ship on Colab; this just pins versions.

In [ ]:
%pip install -q -r exercises/requirements_robustness.txt

**0.3 — 🔧 PROVIDED — visual helpers.** Every card, digit-triptych and result plot lives here so the
teaching cells stay about the *idea*. Run it once, then treat it as a black box.

In [ ]:
#@title 🎨 PROVIDED — visual helpers (run, don't edit) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

PURPLE, PURPLE2 = "#667eea", "#764ba2"
GREEN, RED, AMBER, BLUE = "#39b36a", "#e0796d", "#e0a23c", "#4c8dd8"
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})

def explain(title, body, tone="info", icon="💡"):
    # Themed explanation card; `body` is raw HTML.
    edge = {"info": PURPLE, "warn": AMBER, "risk": RED, "good": GREEN}[tone]
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f7f8ff,#fbf5ff);
     border:1px solid #ecebff;border-left:5px solid {edge};border-radius:14px;padding:15px 18px;margin:8px 0;max-width:860px;color:#3a3357">
  <div style="font-size:15.5px;font-weight:800;color:#3b2d6b;margin-bottom:7px">{icon} {title}</div>
  <div style="font-size:13px;color:#3a3357;line-height:1.6">{body}</div></div>'''))

def _img(ax, x, title, color="#2c2350"):
    ax.imshow(np.asarray(x).reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=11, color=color, pad=6)
    ax.set_xticks([]); ax.set_yticks([])

def show_attack(x_clean, x_adv, p_clean, p_adv, conf_clean, conf_adv, eps, target=None):
    # The signature triptych: clean | perturbation (amplified) | adversarial.
    x_clean = np.asarray(x_clean).reshape(28, 28)
    x_adv   = np.asarray(x_adv).reshape(28, 28)
    delta   = x_adv - x_clean
    ok = (p_adv != p_clean) if target is None else (p_adv == target)
    fig, ax = plt.subplots(1, 3, figsize=(8.4, 3.1))
    _img(ax[0], x_clean, f"clean  ->  {p_clean}   ({conf_clean:.0%})", GREEN)
    m = np.abs(delta).max() + 1e-9
    ax[1].imshow(delta, cmap="bwr", vmin=-m, vmax=m)
    ax[1].set_title(f"perturbation (amplified)\nL-inf = {np.abs(delta).max():.3f}  (<= eps={eps})",
                    fontsize=10, color="#6b6685", pad=6)
    ax[1].set_xticks([]); ax[1].set_yticks([])
    _img(ax[2], x_adv, f"adversarial  ->  {p_adv}   ({conf_adv:.0%})", RED if ok else GREEN)
    headline = ("FOOLED  —  " + (f"forced into {target}" if target is not None else "misclassified")) \
               if ok else "SURVIVED  —  still correct"
    fig.suptitle(headline, fontsize=13, fontweight="bold", color=(RED if ok else GREEN), y=1.02)
    plt.tight_layout(); plt.show()

def robustness_curve(eps_list, series, title, ylabel="accuracy", xlabel="perturbation budget  eps  (L-inf)"):
    # series = {label: (ys, color)} — one line per model/attack.
    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    for lbl, (ys, col) in series.items():
        ax.plot(eps_list, ys, marker="o", ms=5, lw=2.2, color=col, label=lbl)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold", color="#2c2350")
    ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=10)
    plt.tight_layout(); plt.show()

def logit_bounds(names, lb, ub, point, y_true, certified):
    # Show per-class output intervals [lb, ub] from the box, vs the true class.
    fig, ax = plt.subplots(figsize=(7.6, 3.6))
    xs = np.arange(len(names))
    centers = (lb + ub) / 2
    ax.errorbar(xs, centers, yerr=[centers - lb, ub - centers], fmt="none",
                ecolor="#b9a9e6", elinewidth=8, capsize=0, alpha=0.8)
    ax.scatter(xs, point, color=PURPLE2, zorder=5, s=28, label="clean logit")
    ax.axhline(lb[y_true], color=GREEN, ls="--", lw=1.6, label=f"true-class ({y_true}) lower bound")
    ax.scatter([y_true], [point[y_true]], color=GREEN, s=60, zorder=6)
    ax.set_xticks(xs); ax.set_xticklabels(names)
    ax.set_xlabel("class"); ax.set_ylabel("logit interval over the eps-box")
    verdict = "CERTIFIED  —  every rival's ceiling is below the true class's floor" if certified \
              else "NOT CERTIFIED  —  some rival interval overlaps the true class"
    ax.set_title(verdict, fontweight="bold", color=(GREEN if certified else AMBER))
    ax.legend(frameon=False, fontsize=9.5, loc="lower right"); ax.grid(alpha=0.2, axis="y")
    plt.tight_layout(); plt.show()

def scoreboard(rows, title="Scoreboard"):
    # rows = list of (label, {col: value}) — a themed comparison table.
    cols = list(rows[0][1].keys())
    head = "".join(f'<th style="padding:8px 14px;text-align:right;color:#6b6685;font-size:12px">{c}</th>' for c in cols)
    body = ""
    for lbl, d in rows:
        tds = "".join(f'<td style="padding:8px 14px;text-align:right;font-variant-numeric:tabular-nums;font-weight:700;color:#2c2350">{d[c]}</td>' for c in cols)
        body += f'<tr><td style="padding:8px 14px;font-weight:800;color:#3b2d6b">{lbl}</td>{tds}</tr>'
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:#fff;border:1px solid #ecebff;border-radius:14px;padding:8px 6px;margin:8px 0;max-width:640px;color:#2c2350">
  <div style="font-weight:800;color:#3b2d6b;padding:8px 14px;font-size:15px">{title}</div>
  <table style="border-collapse:collapse;width:100%"><tr><th></th>{head}</tr>{body}</table></div>'''))

def quiz(question, options, answer_index, reveal, title="🧠 Quick check"):
    # Self-contained multiple-choice: click an option, then check. Answer lives in a JS blob.
    import json as _json
    data = {"ans": int(answer_index), "reveal": reveal}
    uid = "q_" + str(abs(hash((question, tuple(options), answer_index))) % 10**8)
    rows = "".join('<div class="q-opt" data-i="%d"><span class="q-dot"></span>%s</div>' % (i, o)
                   for i, o in enumerate(options))
    tmpl = r'''
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;border:1px solid #e6e8ee;border-radius:14px;padding:16px;max-width:780px;background:#fff;color:#1c1e2a}
#__UID__ .q-head{font-weight:800;font-size:15px;margin-bottom:4px;color:#2b2d6b}
#__UID__ .q-q{color:#444;font-size:13.5px;margin-bottom:12px;line-height:1.55}
#__UID__ .q-opt{display:flex;align-items:flex-start;gap:10px;border:1px solid #e2e5ef;border-radius:10px;padding:10px 12px;margin-bottom:8px;cursor:pointer;font-size:13.5px;line-height:1.5;transition:.12s;color:#1c1e2a}
#__UID__ .q-opt:hover{border-color:#764ba2;background:#faf9ff}
#__UID__ .q-dot{width:16px;height:16px;border-radius:50%;border:2px solid #c2c7da;flex:0 0 auto;margin-top:2px}
#__UID__ .q-opt code{background:#f3f0ff;border-radius:5px;padding:1px 5px;font-size:12.5px}
#__UID__ .q-opt.sel{border-color:#764ba2;background:#f1edff}
#__UID__ .q-opt.sel .q-dot{background:#764ba2;border-color:#764ba2}
#__UID__ .q-opt.ok{border-color:#46b46e;background:#e7f7ec}
#__UID__ .q-opt.no{border-color:#e07a7a;background:#fdecec}
#__UID__ .q-btn{cursor:pointer;border:none;border-radius:8px;padding:9px 18px;font-size:13.5px;font-weight:700;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);margin-top:6px}
#__UID__ .q-rev{font-size:13px;color:#2c2350;margin-top:10px;min-height:18px;line-height:1.6}
</style>
<div id="__UID__"><div class="q-head">__TITLE__</div><div class="q-q">__Q__</div>__ROWS__
<button class="q-btn">Check my answer</button><div class="q-rev"></div></div>
<script>(function(){
  const D=__DATA__, root=document.getElementById("__UID__");
  const opts=root.querySelectorAll(".q-opt"); let sel=null;
  opts.forEach(o=>o.addEventListener("click",()=>{sel=+o.dataset.i;
    opts.forEach(x=>x.classList.remove("sel","ok","no")); o.classList.add("sel");
    root.querySelector(".q-rev").textContent="";}));
  root.querySelector(".q-btn").addEventListener("click",()=>{
    if(sel===null){root.querySelector(".q-rev").textContent="Pick an option first!";return;}
    opts.forEach(o=>{const i=+o.dataset.i;o.classList.remove("sel");
      if(i===D.ans)o.classList.add("ok"); else if(i===sel)o.classList.add("no");});
    root.querySelector(".q-rev").innerHTML=(sel===D.ans?"✅ Correct. ":"❌ Not quite. ")+D.reveal;});
})();</script>'''
    html = (tmpl.replace("__UID__", uid).replace("__TITLE__", title)
            .replace("__Q__", question).replace("__ROWS__", rows)
            .replace("__DATA__", _json.dumps(data)))
    display(HTML(html))

print("visual helpers ready ✅")

**0.4 — 🔧 PROVIDED — the model and the data.** Our target is a tiny fully-connected net on **MNIST**
(`784 → 200 → ReLU → 10`). Small on purpose: small enough that we can not only *attack* it but later
**certify** it exactly.

The very first layer is a `Normalize` step *inside* the network. That is deliberate: it lets every
attack and every certificate reason about the **real image** in `[0,1]` pixel space, not some
pre-normalised tensor. We load two identically-shaped checkpoints:

- **`model_std`** — trained normally (`net_10_none.pt`).
- **`model_rob`** — trained with **PGD adversarial training** (`net_10_PGD.pt`) — same architecture,
  same data, a robustness-aware loss. Part 4 explains how it was made.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

class Normalize(nn.Module):
    # MNIST mean/std, applied INSIDE the net so attacks live in real [0,1] pixel space.
    def forward(self, x): return (x - 0.1307) / 0.3081

class View(nn.Module):
    def forward(self, x): return x.view(x.shape[0], -1)

def build_net():
    # Sequential so we can walk it layer-by-layer when we certify.
    return nn.Sequential(Normalize(), View(),
                         nn.Linear(28*28, 200), nn.ReLU(), nn.Linear(200, 10))

def load_net(path):
    net = build_net()
    net.load_state_dict(torch.load(path, map_location=device))
    return net.to(device).eval()

model_std = load_net("exercises/models/net_10_none.pt")   # ordinary training
model_rob = load_net("exercises/models/net_10_PGD.pt")    # PGD-adversarial training

# MNIST test set (images already in [0,1]; no external normalize — the net does it).
test_set = datasets.MNIST("exercises/temp/mnist", train=False, download=True,
                          transform=transforms.ToTensor())
X = torch.stack([test_set[i][0] for i in range(1000)]).to(device)   # 1000 test images
Y = torch.tensor([test_set[i][1] for i in range(1000)]).to(device)

@torch.no_grad()
def predict(model, x):
    p = model(x).softmax(1)
    conf, cls = p.max(1)
    return cls, conf

@torch.no_grad()
def accuracy(model, x, y):
    return (model(x).argmax(1) == y).float().mean().item()

print(f"device: {device}")
print(f"clean test accuracy   —   standard {accuracy(model_std, X, Y):.1%}"
      f"   |   robust {accuracy(model_rob, X, Y):.1%}")

---
## 1 · The gradient points *both* ways — FGSM 🗡️

Training a network, you compute the gradient of the loss with respect to the **weights** and step
*downhill* to make the loss small. An attacker computes the gradient of the loss with respect to the
**input pixels** and steps *uphill* to make the loss **large** — while the weights sit frozen.

That single sign flip is the whole idea of the **Fast Gradient Sign Method** (FGSM). One step:

$$x_{\text{adv}} \;=\; \text{clip}_{[0,1]}\Big(x \;+\; \varepsilon \cdot \operatorname{sign}\big(\nabla_x \,\mathcal{L}(f(x),\,y)\big)\Big)$$

- $\nabla_x \mathcal{L}$ — how the loss changes as we wiggle each pixel.
- $\operatorname{sign}(\cdot)$ — we don't care *how much*, only *which way*: push every pixel to its
  most damaging corner.
- $\varepsilon$ — the **budget**: the largest change *any* pixel may take (the $L_\infty$ ball). Keep
  it small and the change is invisible; that is exactly what makes the attack scary.

**1.1 — 🎯 Implement FGSM.** One forward pass, one backward pass to the input, one signed step.

*Toolbox:* `F.cross_entropy` · `torch.autograd.grad(loss, x_adv)` (returns a **1-tuple** — note the `grad,` unpacking) · `.sign()` · `.clamp(0, 1)`.
Untargeted means you want to **raise** the loss on the true label — so which direction does the signed step go, with or against the gradient?

In [ ]:
def fgsm(model, x, y, eps):
    # Untargeted FGSM: one signed gradient step that RAISES the loss on the true label y.
    x_adv = x.clone().detach().requires_grad_(True)
    loss = ???                                   # 🎯 cross-entropy of the logits against the TRUE label y
    grad, = ???                                  # 🎯 gradient of loss w.r.t. the INPUT x_adv  (1-tuple — keep the comma)
    x_adv = ???                                  # 🎯 one signed step that RAISES the loss (size eps)
    return x_adv.clamp(0, 1).detach()            # stay a valid image


**1.2 — Fool one digit.** Pick a cleanly-classified image, craft its adversarial twin at `ε = 0.15`
(about a 15%-of-full-scale nudge per pixel, still visually the same digit), and look at all three
panels: the original, the perturbation (hugely amplified so you can even see it), and the result.

In [ ]:
i = 3                                              # a test image the model gets right
x0, y0 = X[i:i+1], Y[i:i+1]
x_adv = fgsm(model_std, x0, y0, eps=0.15)

c0, cf0 = predict(model_std, x0)
ca, cfa = predict(model_std, x_adv)
show_attack(x0.cpu().numpy(), x_adv.cpu().numpy(),
            c0.item(), ca.item(), cf0.item(), cfa.item(), eps=0.15)
print(f"true label {y0.item()}  —  clean -> {c0.item()} ({cf0.item():.0%})   "
      f"adversarial -> {ca.item()} ({cfa.item():.0%})")

**1.3 — How the budget ε trades off.** One image is an anecdote. Sweep ε across the whole test set and
watch accuracy fall off a cliff — the network that was 99% correct is wrong on most images long before
the change becomes visible.

In [ ]:
eps_grid = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

def batch_accuracy_under(attack, model, x, y, eps):
    x_adv = attack(model, x, y, eps) if eps > 0 else x
    return accuracy(model, x_adv, y)

fgsm_acc = [batch_accuracy_under(fgsm, model_std, X, Y, e) for e in eps_grid]
for e, a in zip(eps_grid, fgsm_acc):
    print(f"  eps = {e:.2f}   accuracy {a:5.1%}")
robustness_curve(eps_grid, {"standard model · FGSM": (fgsm_acc, RED)},
                 "One gradient step is enough: accuracy vs. attack budget")

In [ ]:
#@title 🧠 Where does the gradient in FGSM come from? { display-mode: "form" }
quiz("FGSM steps along the sign of a gradient. The gradient of *what*, with respect to *what*?",
     ["The loss w.r.t. the model weights — the same gradient training uses",
      "The loss w.r.t. the input pixels, with the weights held fixed",
      "The model's output w.r.t. the label",
      "The accuracy w.r.t. the learning rate"],
     1,
     "Training differentiates the loss w.r.t. <b>weights</b> (and steps downhill). FGSM freezes the "
     "weights and differentiates the loss w.r.t. the <b>input</b>, then steps <i>uphill</i> — same "
     "machinery, opposite target and opposite direction.")

---
## 2 · Untargeted vs. targeted — "just be wrong" vs. "say *this*" 🎯

The FGSM above is **untargeted**: it raises the loss on the true class, so the model lands on
*whatever* wrong label is nearest. Often that is not enough for a real attacker. A fraudster editing a
cheque doesn't want the amount to become *some* other digit — they want it to become a **specific**
one.

That is a **targeted** attack. The change is beautifully small:

| | pushes the loss on … | step direction | outcome |
|---|---|---|---|
| **untargeted** | the **true** class $y$ | **+** gradient (raise it) | any wrong answer |
| **targeted** | the **target** class $t$ | **−** gradient (lower it) | exactly answer $t$ |

To force class $t$ we *minimise* the loss toward $t$ — i.e. step **downhill** on $\mathcal{L}(f(x),t)$,
the same move a trainer would make to *teach* the label $t$, except we edit the image instead of the
weights.

**2.1 — 🎯 Targeted FGSM.** Same shape as 1.1, **two changes**:
1. The loss is measured against the **target** label `t`, not the true label `y`.
2. You now want to *lower* that loss — so the sign of the step flips.

In [ ]:
def fgsm_targeted(model, x, t, eps):
    # Targeted FGSM: one signed step that LOWERS the loss toward the chosen class t.
    x_adv = x.clone().detach().requires_grad_(True)
    loss = ???                                   # 🎯 cross-entropy — against the TARGET t this time, not y
    grad, = torch.autograd.grad(loss, x_adv)
    x_adv = ???                                  # 🎯 one signed step that LOWERS that loss (which sign?)
    return x_adv.clamp(0, 1).detach()


**2.2 — Turn a digit into whatever you like.** Take the image below (a real handwritten digit) and
forge every one of the ten possible verdicts from it. One FGSM step is often too weak to hit an
*arbitrary* target, so we take a few — a first taste of the iteration that becomes PGD in Part 3.

🎯 You only fill the **inner step** of the loop: it's the targeted move from 2.1, but scaled by the small
per-iteration size `step` (already computed) instead of the full `eps` — then `.detach()`.
The two lines beneath it (project into the eps-ball, clamp to valid pixels) are provided.

In [ ]:
def fgsm_targeted_iter(model, x, t, eps, steps=15):
    # A few small targeted steps, each re-projected into the eps-ball around x.
    x_adv = x.clone().detach()
    step = 2.5 * eps / steps
    for _ in range(steps):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), t)
        grad, = torch.autograd.grad(loss, x_adv)
        x_adv = ???                                              # 🎯 same move as 2.1 (subtract step*grad.sign() to go DOWNHILL toward t),
        #                                                        #    but use the small `step` above instead of eps, then wrap it in (...).detach()
        x_adv = x.detach() + (x_adv - x.detach()).clamp(-eps, eps)   # project into the eps-ball
        x_adv = x_adv.clamp(0, 1)
    return x_adv

src = 12                                           # source image
xs, ys = X[src:src+1], Y[src:src+1]
fig, ax = plt.subplots(1, 10, figsize=(13, 1.7))
for t in range(10):
    tgt = torch.tensor([t], device=device)
    xt = fgsm_targeted_iter(model_std, xs, tgt, eps=0.3, steps=15)
    cls, conf = predict(model_std, xt)
    got = cls.item()
    ax[t].imshow(xt.cpu().numpy().reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax[t].set_title(f"->{t}\n{'✓' if got==t else '✗'} {conf.item():.0%}",
                    fontsize=9, color=(GREEN if got==t else RED))
    ax[t].set_xticks([]); ax[t].set_yticks([])
fig.suptitle(f"One real '{ys.item()}' (eps=0.3), forged into all ten targets",
             fontsize=12, fontweight="bold", color="#2c2350", y=1.28)
plt.tight_layout(); plt.show()

In [ ]:
#@title 🧠 What flips an untargeted attack into a targeted one? { display-mode: "form" }
quiz("You have working untargeted FGSM. What is the minimal change to force a *specific* class t?",
     ["Use a much larger eps so the model gets confused",
      "Compute the loss toward t instead of the true label, and flip the step to minus eps*sign(grad)",
      "Run the untargeted attack ten times and keep the run that lands on t",
      "Swap cross-entropy for mean-squared error"],
     1,
     "Targeted = descend the loss <i>toward</i> the class you want (minus sign), untargeted = ascend "
     "the loss <i>away</i> from the truth (plus sign). Same gradient machinery, retargeted and "
     "reversed. Bigger eps or lucky restarts are neither necessary nor reliable.")

---
## 3 · The strong attack — Projected Gradient Descent (PGD) 🔁

FGSM takes **one** big jump in the sign direction. But the loss surface bends: the best direction at
the start is not the best direction after you've moved. **PGD** fixes this by taking **many small
steps**, and after each one **projecting** back into the $\varepsilon$-ball so the total perturbation
never exceeds the budget:

$$x^{k+1} \;=\; \Pi_{\|x'-x\|_\infty \le \varepsilon}\Big(x^{k} + \alpha\,\operatorname{sign}\big(\nabla_x \mathcal L(f(x^k),y)\big)\Big)$$

- $\alpha$ — a small **step size** (we use $\alpha = 2.5\,\varepsilon / k$, a common choice so the
  steps can traverse the ball a couple of times).
- $\Pi$ — the **projection**: clamp back into the box $[x-\varepsilon,\,x+\varepsilon]$ and into
  $[0,1]$ pixels.
- a **random start** inside the ball helps escape the flat spot right at $x$.

PGD is the workhorse of the field: strong enough that "survives PGD" is the *de facto* empirical test
of robustness, and it doubles as the inner loop of the best-known defense (Part 4).

**3.1 — 🎯 Implement PGD** (untargeted; a `target` argument gives you the targeted variant for free).

PGD = FGSM done `k` times, each step of size `step`, re-projected into the eps-ball after every step.
*Two lines to fill:*
- **direction** — the untargeted case pushes to raise the loss on `y`; when a `target` is supplied you push the other way. One expression using `grad.sign()` and the `target is not None` test.
- **eps-ball projection** — after the step, clamp `x_adv` so it never strays more than `eps` from the *original* `x` in any pixel. Think about the **deviation** `x_adv - x` and `.clamp(-eps, eps)`.

In [ ]:
def pgd(model, x, y, eps, k=20, target=None):
    # L-infinity PGD. Untargeted (raise loss on y) unless `target` is given (lower loss toward it).
    step = 2.5 * eps / k
    # random start inside the eps-ball, clamped to valid pixels
    x_adv = (x + eps * (2 * torch.rand_like(x) - 1)).clamp(0, 1)
    for _ in range(k):
        x_adv = x_adv.detach().requires_grad_(True)
        lbl = target if target is not None else y
        loss = F.cross_entropy(model(x_adv), lbl)
        grad, = torch.autograd.grad(loss, x_adv)
        direction = ???                            # 🎯 sign of the gradient — but flipped when a `target` is given
        x_adv = x_adv.detach() + step * direction
        x_adv = ???                                # 🎯 project back into the eps-ball: keep the deviation from x within ±eps
        x_adv = x_adv.clamp(0, 1)                   # project into valid pixels
    return x_adv.detach()


**3.2 — PGD vs. FGSM, head to head.** Same budget, same images. PGD's extra steps find adversarial
examples exactly where FGSM's single jump overshoots or stalls — the gap between the two curves is the
price of only looking once.

In [ ]:
pgd_acc = [batch_accuracy_under(lambda m,a,b,e: pgd(m,a,b,e,k=20), model_std, X, Y, e)
           for e in eps_grid]
for e, af, ap in zip(eps_grid, fgsm_acc, pgd_acc):
    print(f"  eps = {e:.2f}   FGSM {af:5.1%}   |   PGD {ap:5.1%}")
robustness_curve(eps_grid,
                 {"standard · FGSM (1 step)":  (fgsm_acc, AMBER),
                  "standard · PGD (20 steps)": (pgd_acc,  RED)},
                 "More steps, less mercy: PGD dominates FGSM at every budget")

In [ ]:
#@title 🧠 Why does PGD beat FGSM at the same eps? { display-mode: "form" }
quiz("Both attacks obey the same L-inf budget eps. Why does PGD find adversarial examples FGSM misses?",
     ["PGD is allowed a larger perturbation than FGSM",
      "PGD takes many small steps and re-projects, so it follows the curving loss surface instead of "
      "committing to one direction",
      "PGD uses the true label while FGSM does not",
      "PGD trains the model, FGSM only tests it"],
     1,
     "Same eps, same budget. FGSM commits to the single sign direction at the start point; PGD "
     "re-reads the gradient after each small step and projects back into the ball, so it can navigate "
     "a loss surface that bends. That is strictly more search for the same budget.")

---
## 4 · Fighting back — adversarial training 🛡️

If the attacker finds the worst point in each ε-ball, the natural defense is to **train on that worst
point**. Adversarial training replaces the ordinary loss with a **min–max** objective:

$$\min_{\theta}\;\; \mathbb{E}_{(x,y)}\;\Big[\max_{\|\delta\|_\infty\le\varepsilon}\; \mathcal L\big(f_\theta(x+\delta),\,y\big)\Big]$$

The **inner max** is exactly PGD — for each batch, attack your own model; the **outer min** then
updates the weights to classify *those* adversarial images correctly. It is expensive (every training
step runs a small PGD attack) but remarkably effective. `model_rob` was made this way.

**4.1 — 🔧 PROVIDED — the recipe** that produced `net_10_PGD.pt`. Reading it is the point; you do
**not** need to run it (a full robust train on CPU is slow — the pretrained checkpoint is already
loaded). This is just PGD wrapped around an ordinary training loop.

In [ ]:
# ---- how model_rob was trained (reference; the checkpoint is already loaded) ----
def adversarial_train(model, loader, epochs, eps, k, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            x_adv = pgd(model, xb, yb, eps=eps, k=k)         # inner max: attack our own model
            opt.zero_grad()
            F.cross_entropy(model(x_adv), yb).backward()     # outer min: fit the adversarial images
            opt.step()
    return model
# model_rob = adversarial_train(build_net().to(device), train_loader, epochs=10, eps=0.1, k=7)
print("recipe defined (not executed — model_rob is the pretrained result)")

**4.2 — Does it hold up?** Re-run the PGD attack on the robust model and overlay it on the standard
model's curve. The robust model gives up a sliver of *clean* accuracy in exchange for a dramatically
flatter collapse under attack.

In [ ]:
pgd_acc_rob = [batch_accuracy_under(lambda m,a,b,e: pgd(m,a,b,e,k=20), model_rob, X, Y, e)
               for e in eps_grid]
robustness_curve(eps_grid,
                 {"standard model · PGD": (pgd_acc,     RED),
                  "robust model · PGD":   (pgd_acc_rob, GREEN)},
                 "Adversarial training earns a much flatter robustness curve")

scoreboard([
    ("standard  (net_10_none)", {"clean acc": f"{accuracy(model_std, X, Y):.1%}",
                                 "PGD@0.1":  f"{pgd_acc[eps_grid.index(0.10)]:.1%}"}),
    ("robust  (net_10_PGD)",    {"clean acc": f"{accuracy(model_rob, X, Y):.1%}",
                                 "PGD@0.1":  f"{pgd_acc_rob[eps_grid.index(0.10)]:.1%}"}),
], title="Standard vs. adversarially-trained")

In [ ]:
#@title 🧠 The robust model survives 20-step PGD. Is it *proven* safe? { display-mode: "form" }
quiz("The robust model shrugs off 20-step PGD at eps=0.1. Can we conclude it is safe against every "
     "perturbation with norm(delta, inf) <= 0.1?",
     ["Yes — PGD is the strongest attack, so surviving it proves safety",
      "No — an attack that finds nothing only means *this* attack failed; a stronger or luckier one "
      "might still succeed",
      "Yes, but only for the images we happened to test",
      "No, because clean accuracy dropped"],
     1,
     "An attack is a <b>lower</b> bound on vulnerability: it can prove a model <i>broken</i> (here is "
     "the counterexample) but never prove it <i>safe</i> — absence of a found attack is not absence "
     "of an attack. To claim safety we need to reason over the <i>whole</i> eps-ball at once. That is "
     "certification. →")

---
## 5 · From "couldn't break it" to "can't be broken" — certification 📐

Every attack so far tries **one point at a time** and hopes to get lucky. Certification flips the
question: instead of a single image $x$, push the **entire ε-box** $[x-\varepsilon,\,x+\varepsilon]$
through the network *at once*, and track an over-approximation of everything it could become. If, at
the output, the true class's **worst case** still beats every other class's **best case**, then **no**
input in that box — no attack, however clever — can change the prediction. That is a proof.

The simplest such method is **interval bound propagation** (the "**box**"). We keep a lower/upper
bound `[l, u]` for every neuron and push it layer by layer:

- **Affine `z = Wx + b`** — split into center $c=\tfrac{l+u}{2}$ and radius $r=\tfrac{u-l}{2}$. The
  center maps by $W$; the radius grows by $|W|$ (magnitudes, because a negative weight flips which
  side is which): $c' = Wc+b,\; r' = |W|\,r$, then $[c'-r',\,c'+r']$.
- **ReLU** — monotonic, so just apply it to both bounds: $[\max(0,l),\,\max(0,u)]$.

Cheap, sound, and — because ReLU is monotone and our net is shallow — tight enough to certify a real
fraction of inputs.

**5.1 — 🎯 Propagate a box** through the `Sequential` net, one layer at a time.

Each activation is an **interval** `[lb, ub]` now, not a point. For a `Linear` layer `Wx + b`, work with the box's
**center** `(lb+ub)/2` and **radius** `(ub-lb)/2` (both already computed for you):
- the center transforms like an ordinary forward pass through the layer;
- the radius can only **grow**, and by how much depends on the *magnitudes* of the weights — think about why a negative weight can't shrink an interval, and which version of `W` that implies.

*Toolbox:* `layer.weight`, `.t()`, `.abs()`, `layer.bias`, `.clamp(0, 1)` for the input box. Then `lb, ub = center - radius, center + radius`.

In [ ]:
@torch.no_grad()
def propagate_box(model, x, eps):
    # Interval bound propagation: return (lb, ub) on the 10 output logits over the eps-box around x.
    lb = ???                        # 🎯 lower corner of the input box, kept to valid pixels
    ub = ???                        # 🎯 upper corner of the input box, kept to valid pixels
    for layer in model:
        if isinstance(layer, (Normalize, View, nn.ReLU)):
            lb, ub = layer(lb), layer(ub)          # monotone maps: apply to both bounds
        elif isinstance(layer, nn.Linear):
            center = (lb + ub) / 2
            radius = (ub - lb) / 2
            c = ???                 # 🎯 push the center through the linear layer (an ordinary forward pass)
            r = ???                 # 🎯 grow the radius through the layer — but which version of W keeps it non-negative?
            lb, ub = c - r, c + r
        else:
            raise NotImplementedError(type(layer))
    return lb, ub

@torch.no_grad()
def certify(model, x, y, eps):
    # Provably robust at x if the true class's lower bound exceeds every rival's upper bound.
    lb, ub = propagate_box(model, x, eps)
    lb, ub = lb[0], ub[0]
    true_lb = lb[y.item()]
    rivals  = torch.cat([ub[:y.item()], ub[y.item()+1:]])
    return bool((true_lb > rivals).all()), lb, ub

**5.2 — See a certificate.** For one image we draw the ten output **intervals** the box produces.
Certified means the green dashed line (the true class's floor) sits **above every other bar's
ceiling** — there is simply no room in the box for another class to win.

In [ ]:
i = 3
xi, yi = X[i:i+1], Y[i:i+1]
ok, lb, ub = certify(model_rob, xi, yi, eps=0.02)
with torch.no_grad():
    point = model_rob(xi)[0].cpu().numpy()
logit_bounds([str(d) for d in range(10)], lb.cpu().numpy(), ub.cpu().numpy(),
             point, yi.item(), ok)
print(f"image {i}: true label {yi.item()}  —  certified robust at eps=0.02?  {ok}")

**5.3 — The whole picture: attack vs. certificate.** Now put both sides on one axis, for both models.
Two truths per model:

- **certified accuracy** (a *lower* bound on robustness): fraction we can **prove** safe.
- **PGD accuracy** (an *upper* bound on robustness): fraction no attack could break.

The truth lives **between** them: below the certified line is *provably safe*, above the PGD line is
*provably broken*, and the band in between is the honest "we don't know yet" — the room where better
attacks *or* better certifiers still compete. Notice the robust model widens the certifiable region by
an order of magnitude.

In [ ]:
cert_eps = [0.0, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05]

@torch.no_grad()
def certified_accuracy(model, x, y, eps):
    if eps == 0:
        return accuracy(model, x, y)
    lb, ub = propagate_box(model, x, eps)                 # batched box
    true_lb = lb.gather(1, y[:, None]).squeeze(1)
    rivals  = ub.clone()
    rivals.scatter_(1, y[:, None], float("-inf"))         # ignore the true class among the rivals
    proved  = (true_lb > rivals.max(1).values)
    correct = (model(x).argmax(1) == y)
    return (proved & correct).float().mean().item()

Xs, Ys = X[:300], Y[:300]                                  # a subset keeps the sweep snappy
cert_std = [certified_accuracy(model_std, Xs, Ys, e) for e in cert_eps]
cert_rob = [certified_accuracy(model_rob, Xs, Ys, e) for e in cert_eps]
pgd_std_c = [batch_accuracy_under(lambda m,a,b,e: pgd(m,a,b,e,k=20), model_std, Xs, Ys, e) for e in cert_eps]
pgd_rob_c = [batch_accuracy_under(lambda m,a,b,e: pgd(m,a,b,e,k=20), model_rob, Xs, Ys, e) for e in cert_eps]

robustness_curve(cert_eps, {
    "standard · PGD (upper bnd)":     (pgd_std_c, "#f0b6ac"),
    "standard · certified (lower)":   (cert_std,  RED),
    "robust · PGD (upper bnd)":       (pgd_rob_c, "#a9dcbc"),
    "robust · certified (lower)":     (cert_rob,  GREEN),
}, title="The robustness sandwich: proven-safe <= truth <= no-attack-found")
scoreboard([
    ("standard", {"cert@0.01": f"{cert_std[cert_eps.index(0.01)]:.0%}",
                  "cert@0.02": f"{cert_std[cert_eps.index(0.02)]:.0%}"}),
    ("robust",   {"cert@0.01": f"{cert_rob[cert_eps.index(0.01)]:.0%}",
                  "cert@0.02": f"{cert_rob[cert_eps.index(0.02)]:.0%}"}),
], title="Certified accuracy (provably robust fraction)")

In [ ]:
#@title 🧠 What exactly does a box certificate guarantee? { display-mode: "form" }
quiz("certify(model, x, y, 0.02) returns True. What have we established?",
     ["The model is correct on x and on the specific PGD examples we tried near it",
      "No input within norm(delta, inf) <= 0.02 of x can change the prediction — for any attack, "
      "present or future",
      "The model is robust on every MNIST image at eps=0.02",
      "The model's clean accuracy is at least 98%"],
     1,
     "The box over-approximates <i>everything</i> the eps-ball around x could become, so a True "
     "verdict rules out <b>all</b> perturbations within it — no attacker, however clever, can win "
     "there. It says nothing about <i>other</i> images (5.3 sweeps those), and because the box is an "
     "over-approximation a False verdict is <i>inconclusive</i>, not proof that an attack exists.")

---
## 🏁 What you built

You worked both sides of adversarial robustness on one small network:

| | you implemented | it answers | direction of the bound |
|---|---|---|---|
| **FGSM** | one signed input step | can I fool it *at all*? | upper bound on robustness |
| **Targeted FGSM** | descend toward class *t* | can I choose the *lie*? | — |
| **PGD** | iterate + project in the ε-ball | the *strong* attack | tighter upper bound |
| **Adversarial training** | min–max (PGD inside the loop) | can I *defend*? | raises true robustness |
| **Box certification** | interval bounds, layer by layer | can I *prove* safety? | lower bound on robustness |

**The one idea to keep.** An **attack** and a **certificate** bound the truth from opposite sides. An
attack that finds nothing is *not* a safety proof — only reasoning over the **entire** ε-ball is.
Adversarial training and tighter certifiers each push the two bounds together, and the gap that
remains is exactly the frontier of the field.

> **Going further (beyond this notebook).** The box is the *loosest* sound certifier — it ignores how
> a neuron's lower and upper bounds are correlated. Tighter relaxations (DeepPoly / CROWN, the
> "zonotope" and LP methods) certify far larger ε, and **certified training** (IBP / TRADES) trains
> *for* provability rather than just against attacks. Same two-sided game, sharper tools.

In [ ]:
#@title 🧠 Final check — attack vs. certificate, in one question { display-mode: "form" }
quiz("A teammate says: I ran 1000-step PGD with 50 restarts and found zero adversarial examples, so "
     "the model is certified robust. What is wrong?",
     ["Nothing — that many restarts is equivalent to a certificate",
      "PGD, however strong, only searches point by point; not finding an attack is an upper bound on "
      "vulnerability, never a proof over the whole eps-ball",
      "They should have used FGSM instead",
      "1000 steps is too many and overfits the attack"],
     1,
     "Certification is a property of the <b>whole</b> eps-ball, established by sound over-approximation "
     "(the box) — not by any number of individual attack trials. Stronger attacks tighten the "
     "<i>upper</i> bound on vulnerability; only a certifier gives the <i>lower</i> bound that the "
     "word certified actually means.")